In [3]:
!pip install onnx==1.17.0

In [2]:
from ultralytics import YOLO

model = YOLO(r'..\fish_guard_server\accounts\services\fish_classifier.pt')

In [10]:
model.predict(r'..\fish_guard_server\media\cropped\photo_LuU7O8l_0_9.jpg')


image 1/1 g:\Work\Fish-Guard_2_0\ML\..\fish_guard_server\media\cropped\photo_LuU7O8l_0_9.jpg: 640x640 hexamitosis 0.57, many_fish 0.21, fin_rot 0.10, dermatomycosis 0.06, not_a_fish 0.04, 124.0ms
Speed: 18.6ms preprocess, 124.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)


[ultralytics.engine.results.Results object with attributes:
 
 boxes: None
 keypoints: None
 masks: None
 names: {0: 'dermatomycosis', 1: 'fin_rot', 2: 'healthy', 3: 'hexamitosis', 4: 'many_fish', 5: 'mycobacteriosis', 6: 'not_a_fish', 7: 'oodiniosis', 8: 'plistophorosis'}
 obb: None
 orig_img: array([[[195, 178, 122],
         [195, 175, 117],
         [200, 169, 108],
         ...,
         [150,  91,  35],
         [150,  91,  35],
         [150,  91,  35]],
 
        [[195, 178, 122],
         [194, 174, 116],
         [199, 168, 107],
         ...,
         [150,  91,  35],
         [150,  91,  35],
         [150,  91,  35]],
 
        [[195, 178, 122],
         [194, 174, 116],
         [199, 168, 107],
         ...,
         [150,  91,  35],
         [150,  91,  35],
         [150,  91,  35]],
 
        ...,
 
        [[139,  80,  34],
         [139,  80,  34],
         [139,  80,  34],
         ...,
         [151,  88,  30],
         [151,  88,  30],
         [151,  88,  30]],


In [27]:
model.args

{'task': 'classify',
 'data': './Dataset_autocropped',
 'imgsz': 640,
 'single_cls': False,
 'model': '..\\fish_guard_server\\accounts\\services\\fish_classifier.pt'}

In [25]:
if hasattr(model, 'trainer') and model.trainer:
    hyp = model.trainer.args
    for key, value in vars(hyp).items():
        print(f"{key}: {value}")

In [ ]:
model.export(format='onnx')

Ultralytics 8.3.134  Python-3.11.4 torch-2.5.1+cu121 CPU (12th Gen Intel Core(TM) i5-12400F)
YOLOv8m-cls summary (fused): 42 layers, 15,774,185 parameters, 0 gradients, 41.6 GFLOPs

PyTorch: starting from '..\fish_guard_server\accounts\services\fish_classifier.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9) (30.2 MB)

ONNX: starting export with onnx 1.14.0 opset 19...
ONNX: slimming with onnxslim 0.1.78...
ONNX: export success  4.0s, saved as '..\fish_guard_server\accounts\services\fish_classifier.onnx' (60.2 MB)

Export complete (4.7s)
Results saved to G:\Work\Fish-Guard_2_0\fish_guard_server\accounts\services
Predict:         yolo predict task=classify model=..\fish_guard_server\accounts\services\fish_classifier.onnx imgsz=640  
Validate:        yolo val task=classify model=..\fish_guard_server\accounts\services\fish_classifier.onnx imgsz=640 data=./Dataset_autocropped  
Visualize:       https://netron.app


'..\\fish_guard_server\\accounts\\services\\fish_classifier.onnx'

In [1]:
import onnx

onnx_model = onnx.load(r'./fish_classifier.onnx')
onnx.checker.check_model(onnx_model)

In [1]:
import onnxruntime as ort
import numpy as np
import cv2

providers = ['CPUExecutionProvider']
session = ort.InferenceSession(r'./fish_classifier.onnx', providers=providers)

In [2]:
input_name = session.get_inputs()[0].name
img = r'..\fish_guard_server\media\cropped\photo_LuU7O8l_0_9.jpg'

In [3]:
import ast
metadata = session.get_modelmeta().custom_metadata_map
names_str = metadata["names"]

class_names = ast.literal_eval(names_str)

print(class_names)
print(class_names[2])  # healthy

{0: 'dermatomycosis', 1: 'fin_rot', 2: 'healthy', 3: 'hexamitosis', 4: 'many_fish', 5: 'mycobacteriosis', 6: 'not_a_fish', 7: 'oodiniosis', 8: 'plistophorosis'}
healthy


In [ ]:
def preprocess(image, img_size):
    h, w = image.shape[:2]
    
    # Letterbox
    scale = min(img_size / w, img_size / h)
    new_w, new_h = int(w * scale), int(h * scale)
    
    resized = cv2.resize(image, (new_w, new_h))
    
    # Создаем холст
    canvas = np.full((img_size, img_size, 3), 114, dtype=np.uint8)
    y_offset = (img_size - new_h) // 2
    x_offset = (img_size - new_w) // 2
    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    
    # Преобразуем для модели
    image_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
    image_rgb = image_rgb.astype(np.float32) / 255.0
    
    # Транспонируем в формат (1, 3, H, W)
    input_tensor = image_rgb.transpose(2, 0, 1)[np.newaxis, ...]
    
    return input_tensor, (w, h), (x_offset, y_offset), scale

def predict(image, session, img_size=640):
    input_tensor, original_shape, offset, scale = preprocess(image, img_size=img_size)
    outputs = session.run(None, {input_name: input_tensor})
    return outputs

In [ ]:
input_image = cv2.imread(img)
input_tensor, original_shape, offset, scale = preprocess(input_image, img_size=640)

array([[[[0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883],
         [0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883],
         [0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883],
         ...,
         [0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883],
         [0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883],
         [0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883]],

        [[0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883],
         [0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883],
         [0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.44705883],
         ...,
         [0.44705883, 0.44705883, 0.44705883, ..., 0.44705883,
          0.44705883, 0.4

In [18]:
input_tensor[0][0][100][100]

0.26666668

In [10]:
input_image = cv2.imread(img) 

probs = predict(input_image, session=session)[0][0]
class_id = int(np.argmax(probs))
confidence = float(probs[class_id])

print(class_id, confidence)
print(class_names[class_id])

3 0.571463942527771
hexamitosis


In [24]:
predict(input_image, session=session)[0].shape

(1, 9)

In [54]:
session.get_inputs()[0].name

'images'

In [ ]:
[output.name for output in session.get_outputs()]

['output0']

In [ ]:
li = [0, 1, 2]

In [50]:
try:
    metadata = session.get_modelmeta().custom_metadata_map
    print("Custom metadata:", metadata)
    if 'names' in metadata:
        print("Нашел классы в модели!")
except:
    print("Нет метаданных")

Custom metadata: {'date': '2026-01-01T18:07:34.953537', 'description': 'Ultralytics YOLOv8m-cls model trained on ./Dataset_autocropped', 'author': 'Ultralytics', 'version': '8.3.134', 'task': 'classify', 'license': 'AGPL-3.0 License (https://ultralytics.com/license)', 'docs': 'https://docs.ultralytics.com', 'stride': '1', 'batch': '1', 'imgsz': '[640, 640]', 'names': "{0: 'dermatomycosis', 1: 'fin_rot', 2: 'healthy', 3: 'hexamitosis', 4: 'many_fish', 5: 'mycobacteriosis', 6: 'not_a_fish', 7: 'oodiniosis', 8: 'plistophorosis'}", 'args': "{'batch': 1, 'half': False, 'dynamic': False, 'simplify': True, 'opset': None, 'nms': False}", 'channels': '3'}
Нашел классы в модели!


In [52]:
metadata = session.get_modelmeta().custom_metadata_map
names_str = metadata['names']  # Это строка!

print(names_str)

{0: 'dermatomycosis', 1: 'fin_rot', 2: 'healthy', 3: 'hexamitosis', 4: 'many_fish', 5: 'mycobacteriosis', 6: 'not_a_fish', 7: 'oodiniosis', 8: 'plistophorosis'}


In [5]:
# X is numpy array on cpu
ortvalue = ort.OrtValue.ortvalue_from_numpy(X)
ortvalue.device_name()  # 'cpu'
ortvalue.shape()        # shape of the numpy array X
ortvalue.data_type()    # 'tensor(float)'
ortvalue.is_tensor()    # 'True'
np.array_equal(ortvalue.numpy(), X)  # 'True'

# ortvalue can be provided as part of the input feed to a model
session = ort.InferenceSession(
        'model.onnx',
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
results = session.run(["Y"], {"X": ortvalue})
results

NameError: name 'X' is not defined

In [5]:
model.export(format='onnx',
             batch=1,
             device='cpu',
             imgsz=960, simplify=False)

Ultralytics 8.3.134  Python-3.11.4 torch-2.5.1+cu121 CPU (12th Gen Intel Core(TM) i5-12400F)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from 'yolo.pt' with input shape (1, 3, 960, 960) BCHW and output shape(s) (1, 5, 18900) (21.5 MB)
ERROR ONNX: export failure 1.3s: DLL load failed while importing onnx_cpp2py_export: A dynamic link library (DLL) initialization routine failed.


ImportError: DLL load failed while importing onnx_cpp2py_export: A dynamic link library (DLL) initialization routine failed.